# 05b — rank ablation: r=16/α=32 vs the study's r=32/α=64

**Runtime: L4.** ~20 min, ~2 units. **Run AFTER notebook 05** (its cross-seed
spread is the noise ruler this verdict needs).

One knob turns: LoRA rank 32 → 16 with α kept at 2·r (same α/r scale = 2, so
same update 'volume', half the capacity — 18.5M vs 36.9M trainable params).
Everything else is the frozen winning recipe: no-trace targets, seed 3407,
1 epoch, same mixture, same dev eval.

**Prediction on record:** within cross-seed noise of r=32. If so, the write-up
line is 'rank didn't matter; epochs did.' This is a side ablation — the study's
config stays r=32/α=64 regardless.

In [ ]:
# Setup: Drive, fresh repo, data, routing
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
train_bugs = [b for b in bugs if b['split'] == 'train']
dev_bugs = [b for b in bugs if b['split'] == 'dev']
print(len(train_bugs), 'train |', len(dev_bugs), 'dev')

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Winning mixture, seed 3407 — byte-identical to notebook 04's
SEED = 3407
rng = random.Random(SEED)

def bug_example(b):
    return {'prompt': build_repair_prompt(b['buggy'], b['test_list']),
            'answer': '```python\n' + b['fixed'].strip() + '\n```'}

mixture = []
for b in train_bugs:
    copies = 3 if routing.get(b['id']) == 'sft' else 1
    mixture += [bug_example(b)] * copies
clean_train = [r for r in restraint if r['split'] == 'train']
rng.shuffle(clean_train)
for r in clean_train[:150]:
    mixture.append({'prompt': build_repair_prompt(r['code'], r['test_list']),
                    'answer': '```python\n' + r['code'].strip() + '\n```'})
rng.shuffle(mixture)
print(len(mixture), 'examples')

In [ ]:
# Model with THE ONE CHANGE: r=16, alpha=32 (alpha/r still = 2)
from unsloth import FastLanguageModel
import torch
model, tok = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=1024,
    load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)

from datasets import Dataset
def to_text(ex):
    msgs = [{'role': 'user', 'content': ex['prompt']},
            {'role': 'assistant', 'content': ex['answer']}]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}
ds = Dataset.from_list(mixture).map(to_text)

from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
FastLanguageModel.for_training(model)
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
    dataset_text_field='text',
    args=SFTConfig(per_device_train_batch_size=8, gradient_accumulation_steps=2,
        num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine',
        warmup_ratio=0.05, logging_steps=10, seed=SEED, output_dir='/content/out',
        report_to='none', save_strategy='no'))
trainer = train_on_responses_only(trainer,
    instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()
model.save_pretrained(f'{PHASE3}/sft_notrace_r16_s{SEED}_ep1')
print('trained and saved')

In [ ]:
# Dev eval (same protocol) + VERDICT against r=32 rows from Drive
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor

def passes(code, rec, timeout=6):
    prog = '\n'.join(list(rec['test_imports']) + [code] + list(rec['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

@torch.no_grad()
def dev_eval(tag, k=16):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in dev_bugs:
        text = tok.apply_chat_template(
            [{'role': 'user', 'content': build_repair_prompt(b['buggy'], b['test_list'])}],
            tokenize=False, add_generation_prompt=True)
        enc = tok([text], return_tensors='pt', padding=True, padding_side='left').to('cuda')
        out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                             num_return_sequences=k, max_new_tokens=512,
                             pad_token_id=tok.eos_token_id)
        gen = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        codes = [extract_code(t) for t in gen]
        with ThreadPoolExecutor(max_workers=4) as pool:
            flags = list(pool.map(lambda c: passes(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    p16 = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, 'pass@16': p16, 'gap': p16 - p1, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE3}/dev_eval_{tag}.json', 'w'))
    return res

r16 = dev_eval('r16_ep1_seed3407')
print(f"{'tag':<16} params   pass@1   pass@16   gap")
rows = []
for s in (3407, 42, 1234):
    try:
        r = json.load(open(f'{PHASE3}/dev_eval_ep1_seed{s}.json'))
        rows.append(r)
        print(f"r32 seed{s:<6} 36.9M  {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
    except FileNotFoundError:
        print(f'r32 seed{s}: not on Drive yet (run notebook 05 first for the noise ruler)')
print(f"r16 seed3407   18.5M  {r16['pass@1']*100:6.1f}%  {r16['pass@16']*100:6.1f}%  {r16['gap']*100:5.1f}")
if len(rows) >= 2:
    vals = [r['pass@1']*100 for r in rows]
    spread = max(vals) - min(vals)
    diff = abs(r16['pass@1']*100 - sum(vals)/len(vals))
    print(f"\nr32 cross-seed pass@1 spread: {spread:.1f} pts | r16 vs r32-mean: {diff:.1f} pts")
    print('VERDICT:', 'within noise — rank did not matter (prediction held)' if diff <= spread
          else 'OUTSIDE noise — rank mattered; report which way and rethink nothing else')
print('\nBring this printout back. Study config stays r=32/α=64 either way.')